# Automated Fund Reporting: Factsheets for a Whole Fund Range in Seconds

Every month, asset managers publish a factsheet for each fund and share class:
performance over standard periods, risk statistics, top holdings, allocation and a
manager comment. Done by hand, it means copying numbers between systems, Excel and
PowerPoint, with a risk of error at every step.

This notebook automates the whole chain for three fictitious funds:

1. **Analytics** computed from NAV and holdings data
2. **Charts** generated in code
3. A **commentary written from the numbers** (rule-based, so every sentence is traceable)
4. A two-page **PDF factsheet** and an **Excel data pack** per fund

In [1]:
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("factsheet.py"):
    subprocess.run(["git", "clone", "-q", "https://github.com/adamrejeb86-del/-finance-portfolio.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "reportlab"], check=True)
    os.chdir("-finance-portfolio/24-fund-reporting-automation")

import time, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from factsheet import FundInfo, simulate_fund, period_returns, monthly_table, risk_stats, commentary, build_factsheet

plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})
NAVY, BORDEAUX, GREY = "#1f3b5c", "#7a1f2b", "#8a8a8a"
funds = {
    "europe_quality": (FundInfo("Aurora Europe Quality", "European large-cap equities, quality bias", "Euro Stoxx 50 NR (illustrative)",
                                "I EUR - accumulating", "2022-01-03", 640, "0.75% management fee", "Portfolio management team"),
                       dict(seed=12, alpha=0.025, beta=0.92, te=0.035)),
    "small_mid": (FundInfo("Aurora Euro Small & Mid", "European small and mid caps", "MSCI Europe Small Cap (illustrative)",
                           "R EUR - distributing", "2022-01-03", 210, "1.60% management fee", "Portfolio management team"),
                  dict(seed=8, alpha=0.005, beta=1.10, te=0.06)),
    "global_div": (FundInfo("Aurora Global Dividend", "Global equities, dividend income", "MSCI World High Dividend (illustrative)",
                            "I EUR - distributing", "2022-01-03", 1_150, "0.65% management fee", "Portfolio management team"),
                   dict(seed=23, alpha=-0.004, beta=0.80, te=0.03)),
}

## 1. The analytics behind a factsheet

Shown here for the first fund. Period returns are cumulative for periods under one year
and annualised beyond; risk statistics use three years of daily returns against the benchmark.

In [2]:
info, params = funds["europe_quality"]
nav, bnav, holdings = simulate_fund(**params)
perf, bperf = period_returns(nav), period_returns(bnav)
display(pd.DataFrame({"Fund": perf, "Benchmark": bperf, "Relative": perf - bperf}).T.style.format("{:+.2%}"))
display(risk_stats(nav, bnav).to_frame("3 years").T.round(3))
monthly_table(nav).style.format(lambda v: "" if pd.isna(v) else f"{v:+.1%}")

,1M,3M,YTD,1Y,3Y p.a.,Inception p.a.
Fund,+1.42%,+8.37%,+12.36%,+19.51%,+10.90%,+6.53%
Benchmark,+1.12%,+5.66%,+4.77%,+12.87%,+5.90%,+4.86%
Relative,+0.30%,+2.71%,+7.59%,+6.63%,+5.00%,+1.67%


,Volatility,Benchmark volatility,Sharpe ratio,Beta,Tracking error,Information ratio,Max drawdown
3 years,0.146,0.156,0.719,0.913,0.036,1.168,-0.118


month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Year
year,,,,,,,,,,,,,
2022,+1.4%,+2.5%,-3.7%,+1.9%,-2.3%,-6.0%,+3.6%,+0.9%,+4.6%,+0.2%,+4.2%,-1.6%,+5.2%
2023,+4.9%,-0.2%,-3.7%,+0.4%,-1.8%,+1.2%,+4.3%,-5.4%,+1.3%,-5.4%,-3.4%,-1.5%,-9.4%
2024,+0.9%,+2.2%,-0.4%,+6.3%,+5.0%,-2.6%,+0.2%,-2.0%,-3.6%,+2.3%,+4.3%,-1.1%,+11.4%
2025,-1.3%,+6.6%,-2.7%,-2.5%,+5.7%,-0.2%,-0.3%,-2.6%,+3.7%,+3.4%,+9.6%,-4.4%,+15.1%
2026,-3.1%,+4.0%,-0.4%,+5.4%,-6.4%,+4.1%,+0.6%,+8.7%,-1.8%,+1.4%,,,+12.4%


## 2. A commentary written from the data

Rather than a free-text generator that could invent a reason for a move, the commentary is
assembled from rules: relative performance, top and bottom contributors, best and worst
sectors, year-to-date figures. A compliance officer can check every number.

In [3]:
print(commentary(info, perf, bperf, holdings))

Over the month, Aurora Europe Quality returned +1.42% against +1.12% for the Euro Stoxx 50 NR (illustrative), and outperformed by 30 bp. The main contributors were Company D4 (+43 bp) and Company A6 (+25 bp), while Company A1 cost -48 bp. By sector, Materials added the most and Healthcare detracted the most. Year to date, the fund is +12.36% against +4.77% for its benchmark.


## 3. Batch production for the whole range

In [4]:
os.makedirs("output", exist_ok=True)
t0 = time.perf_counter()
summary = {}
for key, (info, params) in funds.items():
    nav, bnav, holdings = simulate_fund(**params)
    res = build_factsheet(info, nav, bnav, holdings, f"output/{key}_factsheet.pdf", f"output/{key}_data.xlsx")
    summary[info.name] = {"1M": res["performance"]["1M"], "YTD": res["performance"]["YTD"], "1Y": res["performance"]["1Y"],
                          "Relative 1Y": res["performance"]["1Y"] - res["benchmark"]["1Y"], "Sharpe (3y)": res["risk"]["Sharpe ratio"]}
elapsed = time.perf_counter() - t0
print(f"{len(funds)} factsheets (PDF) and data packs (Excel) produced in {elapsed:.1f} seconds:", sorted(os.listdir("output")))
pd.DataFrame(summary).T.style.format({"1M": "{:+.2%}", "YTD": "{:+.2%}", "1Y": "{:+.2%}", "Relative 1Y": "{:+.2%}", "Sharpe (3y)": "{:.2f}"})

3 factsheets (PDF) and data packs (Excel) produced in 1.3 seconds: ['europe_quality_data.xlsx', 'europe_quality_factsheet.pdf', 'global_div_data.xlsx', 'global_div_factsheet.pdf', 'small_mid_data.xlsx', 'small_mid_factsheet.pdf']


,1M,YTD,1Y,Relative 1Y,Sharpe (3y)
Aurora Europe Quality,+1.42%,+12.36%,+19.51%,+6.63%,0.72
Aurora Euro Small & Mid,-1.26%,+16.44%,+15.90%,+0.94%,0.45
Aurora Global Dividend,+0.65%,+4.61%,+12.05%,+0.28%,0.77


## 4. The result

The first page of a generated factsheet (converted to an image to display it here; the PDFs
are in the `output/` folder).

In [5]:
if shutil.which("pdftoppm"):
    subprocess.run(["pdftoppm", "-png", "-r", "70", "-f", "1", "-l", "2", "output/europe_quality_factsheet.pdf", "images/factsheet_page"], check=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 9.5))
    for ax, page in zip(axes, ["images/factsheet_page-1.png", "images/factsheet_page-2.png"]):
        ax.imshow(plt.imread(page)); ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("Open output/europe_quality_factsheet.pdf to see the factsheet.")

## Conclusions
- From raw NAV and holdings to a client-ready PDF and an Excel pack, for every fund, in a
  couple of seconds, with no copy-paste and every number traceable to its source.
- The tests check the arithmetic: annualised returns, the monthly table compounding to the
  annual figure, risk statistics, and that the commentary quotes the computed numbers.
- **Next steps:** connect to a real NAV database, add share-class hedging, ESG metrics and
  multi-language output, and schedule the run at month-end with an email to the sales team.